# GOES-16 / 18 / 19 | Interpretación de ceniza volcánica y composiciones RGB
**INSIVUMEH — Práctica de observación satelital**

**Objetivo.** Seleccionar una fecha, un volcán y un intervalo de observación para interpretar la atmósfera, posibles plumas volcánicas y anomalías térmicas mediante los canales del ABI de GOES-16, 18 y 19. El cuaderno consulta datos públicos de NOAA automáticamente: no requiere subir NetCDF, credenciales, Earth Engine ni claves.

**Productos.** True Color y GeoColor **aproximados**; Ash RGB; Fire Temperature RGB; Dust RGB; RGB de SO₂ exploratorio; Night Microphysics; diferencias y temperaturas de brillo infrarrojas. Las imágenes se exportan con explicación espectral y también pueden generarse **animaciones de 1 a 24 horas**.

**Precaución científica.** Un color llamativo no confirma ceniza, SO₂ ni lava. La clasificación requiere contexto, observaciones coincidentes y otros instrumentos. GeoColor de este ejercicio **no reproduce el producto operacional de NOAA/CIRA**.

**Satélite automático.** GOES-16 (Este) hasta 7 de abril de 2025 a las 15:10 UTC; GOES-19 (Este) desde esa fecha. Para coordenadas al oeste de 110° O, el modo automático prioriza GOES-18 (Oeste) desde 4 de enero de 2023. El selector también permite consultar cada satélite por separado. Si se cruza una fecha de transición, el programa conserva el satélite de origen de cada escena.


## Paso 1. Instalar las bibliotecas
Colab ejecuta las celdas con Python. Esta instalación se realiza automáticamente en la sesión, no en tu computadora.

In [ ]:
%pip -q install xarray netCDF4 scipy pyproj rasterio matplotlib numpy pandas ipywidgets boto3 astral pillow geopandas shapely

In [ ]:
from pathlib import Path
import os, re, math, json, zipfile, warnings
from datetime import date, datetime, timezone, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from pyproj import CRS, Transformer
from scipy.ndimage import map_coordinates
from rasterio.transform import from_bounds
import rasterio
from IPython.display import display, FileLink
import ipywidgets as W
from botocore import UNSIGNED
from botocore.config import Config
import boto3
import textwrap
import geopandas as gpd
from shapely.geometry import Point, box
warnings.filterwarnings('ignore', category=RuntimeWarning)
# Selección de satélite: histórico GOES-East y GOES-West.
SAT_ACTUAL='GOES-19'  # Actualizado al seleccionar la escena
BUCKETS={16:'noaa-goes16',18:'noaa-goes18',19:'noaa-goes19'}
GOES19_EAST_START=datetime(2025,4,7,15,10,tzinfo=timezone.utc)
GOES18_WEST_START=datetime(2023,1,4,18,0,tzinfo=timezone.utc)


def _mode_satellite():
    w=globals().get('satelite_modo')
    return w.value if w is not None else 'auto'


def _point_longitude():
    w=globals().get('lon')
    return float(w.value) if w is not None else -90.88074


def _automatic_satellite(timestamp, longitude=None):
    """Elegir la plataforma nominal según fecha UTC y posición, sin mezclar bandas entre satélites."""
    if longitude is None:
        longitude=_point_longitude()
    # Criterio geográfico orientativo: al oeste de 110°W priorizar GOES-West.
    # La disponibilidad real se valida con las coordenadas y la proyección de cada archivo.
    if float(longitude) <= -110.0 and timestamp >= GOES18_WEST_START:
        return 18
    return 16 if timestamp < GOES19_EAST_START else 19


def _candidate_satellites_for_hour(day, hour):
    mode=_mode_satellite()
    if mode != 'auto':
        return [int(mode)]
    start=datetime(day.year,day.month,day.day,int(hour),0,tzinfo=timezone.utc)
    end=start+timedelta(hours=1)
    sats={_automatic_satellite(start)}
    if _automatic_satellite(end)!=_automatic_satellite(start):
        sats.add(_automatic_satellite(end))
    return sorted(sats)


def _satellite_allowed(sat, timestamp):
    mode=_mode_satellite()
    if mode != 'auto': return sat==int(mode)
    return sat==_automatic_satellite(timestamp)


def _satellite_label(row):
    return 'GOES-'+str(row.get('satelite',row.get('satellite','?')))


def _satellite_from_key(key):
    m=re.search(r'_G(16|18|19)_',str(key))
    if not m:
        raise ValueError('No se pudo identificar satélite por nombre de archivo: '+str(key))
    return int(m.group(1))

DATA=Path('/content/GOES19') if Path('/content').exists() else Path.cwd()/'GOES19'
DATA.mkdir(parents=True,exist_ok=True)
OUTPUT=DATA/'resultados';OUTPUT.mkdir(exist_ok=True)

BOUNDARY_CACHE={}
GTM_DEPTS_URL='https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/guatemala-departments.geojson'
WORLD_URL='https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip'

def _load_world_boundaries():
    if 'world' not in BOUNDARY_CACHE:
        gdf=gpd.read_file(WORLD_URL).to_crs('EPSG:4326')
        BOUNDARY_CACHE['world']=gdf
    return BOUNDARY_CACHE['world']

def _load_gtm_departments():
    if 'gtm_depts' not in BOUNDARY_CACHE:
        gdf=gpd.read_file(GTM_DEPTS_URL).to_crs('EPSG:4326')
        BOUNDARY_CACHE['gtm_depts']=gdf
    return BOUNDARY_CACHE['gtm_depts']

def _country_name_col(gdf):
    for c in ['ADMIN','NAME','admin','name','SOVEREIGNT']:
        if c in gdf.columns:
            return c
    return gdf.columns[0]

def draw_reference_boundaries(ax, lat, lon, bounds):
    west,south,east,north=bounds
    view=box(west,south,east,north)
    try:
        world=_load_world_boundaries()
        cname=_country_name_col(world)
        point=Point(lon,lat)
        hit=world[world.contains(point)]
        if hit.empty:
            hit=world[world.intersects(point.buffer(0.25))]
        if not hit.empty:
            country_name=str(hit.iloc[0][cname]).strip().lower()
            # Recortar el contorno al entorno visible para no perder el zoom sobre el área de interés.
            country_clip=hit.copy()
            try:
                country_clip=country_clip.clip(view.buffer(0.8))
            except Exception:
                pass
            if len(country_clip):
                country_clip.boundary.plot(ax=ax,color='black',linewidth=1.1,alpha=0.90,zorder=8)
            else:
                hit.boundary.plot(ax=ax,color='black',linewidth=1.1,alpha=0.90,zorder=8)
            if country_name=='guatemala':
                depts=_load_gtm_departments()
                try:
                    depts=depts.clip(view.buffer(0.8))
                except Exception:
                    pass
                if len(depts):
                    depts.boundary.plot(ax=ax,color='black',linewidth=0.55,alpha=0.50,zorder=7)
    except Exception:
        pass
    # Marcador del volcán o punto de interés.
    ax.scatter([lon],[lat],marker='^',s=130,color='#d7191c',edgecolors='white',linewidths=0.9,zorder=12)
    # Forzar el zoom al recorte solicitado.
    ax.set_xlim(west,east)
    ax.set_ylim(south,north)


VOLCANES={'Fuego':(14.47459,-90.88074),'Santiaguito':(14.73938,-91.56816),'Pacaya':(14.38249,-90.60136),'Personalizado':(14.47459,-90.88074)}

# El catálogo público S3 no requiere credenciales AWS.
def s3_client(): return boto3.client('s3',region_name='us-east-1', config=Config(signature_version=UNSIGNED))

def filename_time(name):
    m=re.search(r'_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})',Path(str(name)).name)
    if not m: return None
    return datetime(int(m[1]),1,1,tzinfo=timezone.utc)+timedelta(days=int(m[2])-1,hours=int(m[3]),minutes=int(m[4]),seconds=int(m[5]))

def s3_list(day, hours, product='ABI-L2-MCMIPF', channel=None):
    """Listar archivos según satélite seleccionado y año/hora; no depende de un bucket fijo."""
    cli=s3_client(); keys=[]
    for hour in sorted(set(int(h) for h in hours if 0 <= int(h) <= 23)):
        prefix=f'{product}/{day.year}/{day.timetuple().tm_yday:03d}/{hour:02d}/'
        for sat in _candidate_satellites_for_hour(day,hour):
            bucket=BUCKETS[sat]
            pages=cli.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefix)
            for page in pages:
                for obj in page.get('Contents',[]):
                    key=obj['Key']
                    if not key.endswith('.nc') or f'_G{sat:02d}_' not in key:
                        continue
                    if channel and not re.search(r'C'+str(channel).zfill(2)+rf'_G{sat:02d}_',key):
                        continue
                    tm=filename_time(key)
                    if tm is None or not _satellite_allowed(sat,tm):
                        continue
                    keys.append({'key':key,'fecha_utc':tm,'MB':round(obj['Size']/1e6,1),
                                 'bucket':bucket,'satelite':sat})
    return sorted(keys,key=lambda row:row['fecha_utc'])

def local_files(product, day=None, hours=None, channel=None, path=DATA):
    files=[]
    for p in Path(path).rglob('*.nc'):
        if product not in p.name: continue
        match=re.search(r'_G(16|18|19)_',p.name)
        if not match: continue
        sat=int(match.group(1))
        tm=filename_time(p.name)
        if tm is None or not _satellite_allowed(sat,tm): continue
        if channel and not re.search(r'C'+str(channel).zfill(2)+rf'_G{sat:02d}_',p.name): continue
        if day is not None and tm.date()!=day:continue
        if hours is not None and tm.hour not in hours:continue
        files.append({'ruta':str(p),'fecha_utc':tm,'MB':round(p.stat().st_size/1e6,1),
                      'bucket':BUCKETS[sat],'satelite':sat})
    return sorted(files,key=lambda r:r['fecha_utc'])

def download_keys(keys,folder=DATA):
    cli=s3_client();paths=[]
    folder=Path(folder);folder.mkdir(exist_ok=True,parents=True)
    for key in keys:
        sat=_satellite_from_key(key)
        path=folder/Path(key).name
        if not path.exists() or path.stat().st_size==0:
            print('Descargando:',path.name)
            cli.download_file(BUCKETS[sat],key,str(path))
        else:print('En caché:',path.name)
        paths.append(path)
    return paths

def geos_transform(ds):
    if 'goes_imager_projection' not in ds: raise ValueError('Archivo sin proyección ABI goes_imager_projection')
    a=ds['goes_imager_projection'].attrs
    h=float(a['perspective_point_height'])
    crs=CRS.from_proj4(f"+proj=geos +h={h} +lon_0={float(a['longitude_of_projection_origin'])} +sweep={a.get('sweep_angle_axis','x')} +a={float(a['semi_major_axis'])} +b={float(a['semi_minor_axis'])} +units=m +no_defs")
    forward=Transformer.from_crs('EPSG:4326',crs,always_xy=True)
    inverse=Transformer.from_crs(crs,'EPSG:4326',always_xy=True)
    return h,forward,inverse

def coord_indexes(v,coords):
    coords=np.asarray(coords,dtype='float64')
    if coords[0]<coords[-1]: return np.interp(v,coords,np.arange(len(coords)),left=np.nan,right=np.nan)
    return np.interp(v,coords[::-1],np.arange(len(coords))[::-1],left=np.nan,right=np.nan)

def region_bounds(la,lo,rkm):
    dy=rkm/111.32; dx=rkm/(111.32*max(.15,math.cos(math.radians(la))))
    return lo-dx,la-dy,lo+dx,la+dy

def resample_local(ds,vars,lat,lon,radius_km=160,n=180):
    """Reproyecta solo ventanas pequeñas alrededor de un volcán; no carga el disco entero en RAM."""
    h,fw,inv=geos_transform(ds)
    west,south,east,north=region_bounds(lat,lon,radius_km)
    xx=np.linspace(west,east,n);yy=np.linspace(north,south,n)
    lon2,lat2=np.meshgrid(xx,yy)
    xm,ym=fw.transform(lon2,lat2)
    ix=coord_indexes(xm/h,ds.x.values);iy=coord_indexes(ym/h,ds.y.values)
    valid=np.isfinite(ix)&np.isfinite(iy)
    if not valid.any(): raise ValueError('Área fuera del campo del satélite elegido o proyección incorrecta')
    x0=max(0,int(np.floor(np.nanmin(ix[valid])))-2);x1=min(len(ds.x),int(np.ceil(np.nanmax(ix[valid])))+3)
    y0=max(0,int(np.floor(np.nanmin(iy[valid])))-2);y1=min(len(ds.y),int(np.ceil(np.nanmax(iy[valid])))+3)
    rows=np.where(valid,iy-y0,-1000);cols=np.where(valid,ix-x0,-1000)
    results={}
    for v in vars:
        if v not in ds:
            results[v]=np.full((n,n),np.nan,dtype='float32');continue
        da=ds[v]
        if 'x' not in da.dims or 'y' not in da.dims:continue
        sl=da.isel(x=slice(x0,x1),y=slice(y0,y1)).transpose('y','x').values.astype('float32')
        sampled=map_coordinates(sl,[rows,cols],order=1,mode='constant',cval=np.nan, prefilter=False)
        sampled[~valid]=np.nan
        results[v]=sampled
    return results,(west,south,east,north)

def norm(a,vmin,vmax,gamma=1):
    x=np.clip((a-vmin)/(vmax-vmin),0,1)
    return x**(1/gamma)

def rgb3(r,g,b):return np.clip(np.stack([r,g,b],axis=-1),0,1).astype('float32')

def save_geotiff(path,arr,bounds):
    a=np.asarray(arr,dtype='float32')
    if a.ndim==3:a=np.moveaxis(a,-1,0)
    if a.ndim==2:a=a[None,...]
    clean=np.where(np.isfinite(a),a,-9999).astype('float32')
    with rasterio.open(path,'w',driver='GTiff',height=a.shape[1],width=a.shape[2],count=a.shape[0],dtype='float32',crs='EPSG:4326',transform=from_bounds(*bounds,a.shape[2],a.shape[1]),nodata=-9999,compress='deflate') as dst:dst.write(clean)

print('Acceso: repositorios públicos NOAA GOES-16, GOES-18 y GOES-19 (sin cuentas ni tokens).')


# Consulta segura de objetos anónimos NOAA/AWS; sin rutas ni archivos que elegir.
def descargar_automaticamente(row):
    cli=s3_client()
    key=row['key']; sat=int(row.get('satelite',_satellite_from_key(key)))
    bucket=row.get('bucket',BUCKETS[sat])
    destino=DATA/Path(key).name
    tam=cli.head_object(Bucket=bucket,Key=key)['ContentLength']
    if destino.exists() and destino.stat().st_size==tam:
        print(f'En caché: GOES-{sat} / {destino.name} ({tam/1e6:.1f} MB)')
        return destino
    parcial=destino.with_suffix(destino.suffix+'.part')
    if parcial.exists():parcial.unlink()
    print(f'Descargando desde {bucket}: {destino.name} ({tam/1e6:.1f} MB)')
    try:
        cli.download_file(bucket,key,str(parcial))
        if parcial.stat().st_size!=tam:
            raise IOError('La descarga no coincide con el tamaño declarado por NOAA')
        parcial.replace(destino)
    except Exception:
        if parcial.exists():parcial.unlink()
        raise
    return destino

def mostrar_descarga(path):
    """El resultado se descarga desde Colab, sin pedir una ruta local."""
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        display(FileLink(str(path),result_html_prefix='Descargar: '))

Acceso: repositorios públicos NOAA GOES-16, GOES-18 y GOES-19 (sin cuentas ni tokens).


## Paso 2. Comprender la mezcla RGB

Cada color de una composición corresponde a una banda o una operación entre bandas. El canal **R** se representa en rojo, **G** en verde y **B** en azul, pero eso **no significa** que la luz medida sea visible: por ejemplo, el canal C15 detecta infrarrojo de 12.3 µm y puede ser asignado a rojo en el RGB de ceniza.

En las gráficas aparece una **ficha bajo cada imagen** que explica los componentes, los posibles rasgos destacados y las limitaciones. En una animación se debe comparar una misma combinación espectral y la misma escala entre fotogramas.

Referencias: [NOAA – atributos espectrales del ABI](https://www.star.nesdis.noaa.gov/goes/abispectralattributes.php), [CIRA – guía Ash RGB](https://rammb.cira.colostate.edu/training/visit/quick_guides/GOES_Ash_RGB.pdf), [Fire Temperature RGB](https://rgb-satellite-community.github.io/RGB-Composite-Recipes/latest/fire-temperature/).

In [ ]:
RGB_HELP={
 'True Color aproximado':('C02 (0.64 µm) a rojo; verde sintético C01/C02/C03; azul C01 (0.47 µm).','Nubes visibles, topografía e indicios visuales de pluma; no diferencia automáticamente ceniza y nube.'),
 'GeoColor aproximado (NO oficial)':('Composición diurna True Color; en la noche C13 (10.3 µm) y contraste C07–C13 (3.9/10.3 µm).','No reproduce las luces urbanas ni la corrección exacta de GeoColor NOAA. Interpretar según iluminación.'),
 'Ash RGB':('R=C15−C13 (12.3−10.3 µm); G=C14−C11 (11.2−8.4 µm); B=C13 (10.3 µm).','Ayuda a distinguir posibles plumas de ceniza (a menudo tonos rosados/magenta); nubes y humedad pueden generar falsos parecidos.'),
 'Fire Temperature RGB':('R=C07 BT (3.9 µm); G=C06 (2.2 µm); B=C05 (1.6 µm).','Focos térmicos pueden destacar en rojo/amarillo/blanco. No mide temperatura de lava ni garantiza una erupción.'),
 'Dust RGB':('R=C15−C13; G=C14−C11; B=C13 (rangos diferentes de Ash RGB).','Resalta diferencias espectrales de aerosoles minerales, pero no confirma ceniza volcánica.'),
 'SO2 RGB':('R=C09−C10 (6.9−7.3 µm); G=C13−C11; B=C07.','Potencial sensibilidad a plumas de SO₂; no cuantifica concentración.'),
 'Night Microphysics RGB':('R=C15−C13; G=C13−C07; B=C13.','Permite observar nubes nocturnas y su fase; información complementaria al Ash RGB.'),
 'Split Window C15−C13':('Diferencia de BT 12.3−10.3 µm, expresada en K.','Una diferencia negativa puede ser compatible con ceniza, pero no basta para identificarla.'),
 'C07 3.9 µm (BT)':('Temperatura de brillo C07 en Kelvin, ventana IR de onda corta.','Apoya evaluación de anomalías térmicas, con contribución solar durante el día.'),
 'C13 10.3 µm (BT)':('Temperatura de brillo C13 en Kelvin, ventana IR de onda larga.','Diferencia nubes frías, superficie y estructuras de la pluma.'),
}

def compute_products(d,scene_time,lat,lon):
    def c(k): return d.get(f'CMI_C{k:02d}',np.full_like(next(iter(d.values())),np.nan))
    day=True
    try:
        from astral import Observer
        from astral.sun import elevation
        day=elevation(Observer(latitude=lat,longitude=lon),scene_time)>-3
    except Exception: day=True
    c1,c2,c3=c(1),c(2),c(3)
    synthG=0.45*c1+0.45*c2+0.10*c3
    true=rgb3(norm(c2,0,1)**(1/2.2),norm(synthG,0,1)**(1/2.2),norm(c1,0,1)**(1/2.2))
    if day:
        geocolor=true.copy()
    else:
        ice=norm(310-c(13),0,95)
        low=np.clip((c(13)-c(7)+8)/25,0,1)
        geocolor=rgb3(ice*0.88,ice*0.92, np.maximum(ice,low*.4))
    ash=rgb3(norm(c(15)-c(13),-6.7,2.6),norm(c(14)-c(11),-6,6.3),norm(c(13)-273.15,-29.55,29.25))
    fire=rgb3(norm(c(7),273,333,gamma=.4),norm(c(6),0,1),norm(c(5),0,.75))
    dust=rgb3(norm(c(15)-c(13),-6.7,2.6),norm(c(14)-c(11),-.5,20,gamma=2.5),norm(c(13)-273.15,-11.95,15.55))
    so2=rgb3(norm(c(9)-c(10),-4,2),norm(c(13)-c(11),-4,5),norm(c(7)-273.15,-30.1,29.8))
    night=rgb3(norm(c(15)-c(13),-6.7,2.6),norm(c(13)-c(7),-3.1,5.2),norm(c(13)-273.15,-29.6,19.5))
    return {
        'True Color aproximado':true,'GeoColor aproximado (NO oficial)':geocolor,'Ash RGB':ash,
        'Fire Temperature RGB':fire,'Dust RGB':dust,'SO2 RGB':so2,'Night Microphysics RGB':night,
        'Split Window C15−C13':c(15)-c(13), 'C07 3.9 µm (BT)':c(7),'C13 10.3 µm (BT)':c(13)
    },day

def plot_layer(title,layer,notes,bounds,scene_time,path):
    fig,ax=plt.subplots(figsize=(8,7.8))
    if layer.ndim==3:
        img=np.nan_to_num(layer,nan=0.0,posinf=0.0,neginf=0.0)
        ax.imshow(img,extent=(bounds[0],bounds[2],bounds[1],bounds[3]),origin='upper',interpolation='nearest')
    else:
        cmap='coolwarm' if 'Split Window' in title else 'inferno'
        if 'Split Window' in title: lo,hi=-10,10
        else: lo,hi=190,340
        im=ax.imshow(np.ma.masked_invalid(layer),extent=(bounds[0],bounds[2],bounds[1],bounds[3]),origin='upper',cmap=cmap,vmin=lo,vmax=hi)
        fig.colorbar(im,ax=ax,label='K (BT)' if 'Split Window' not in title else 'K (diferencia BT)',shrink=.72)
    draw_reference_boundaries(ax, lat.value if 'lat' in globals() else 14.47, lon.value if 'lon' in globals() else -90.88, bounds)
    ax.set_xlim(bounds[0],bounds[2]); ax.set_ylim(bounds[1],bounds[3])
    ax.set_title(f'{title} | {SAT_ACTUAL} | {scene_time:%Y-%m-%d %H:%M} UTC',fontsize=11)
    ax.set_xlabel('Longitud (°)');ax.set_ylabel('Latitud (°)')
    fig.subplots_adjust(bottom=.28)
    fig.text(.07,.18,textwrap.fill(notes[0],width=87),ha='left',va='top',fontsize=9)
    fig.text(.07,.09,textwrap.fill(notes[1],width=87),ha='left',va='top',fontsize=9)
    fig.savefig(path,dpi=135,bbox_inches='tight');plt.close(fig)

def make_gallery(rows,columns=2):
    from PIL import Image,ImageOps,ImageDraw
    pictures=[Image.open(p).convert('RGB') for p in rows]
    if not pictures: return None
    maxw=max(im.width for im in pictures);maxh=max(im.height for im in pictures)
    gap=24; nrows=math.ceil(len(pictures)/columns)
    canvas=Image.new('RGB',(columns*maxw+(columns+1)*gap,nrows*maxh+(nrows+1)*gap),'#e9eef1')
    for i,img in enumerate(pictures):
        x=gap+(i%columns)*(maxw+gap);y=gap+(i//columns)*(maxh+gap)
        canvas.paste(img,(x,y))
    return canvas

In [ ]:
# Guía de interpretación espectral. Longitudes de onda centrales aproximadas del ABI.
FICHAS_RGB={
 'True Color aproximado':{
   'canales':[
     ('R','C02, 0.64 µm: rojo visible; resalta las nubes brillantes y el terreno iluminado.'),
     ('G','Combinación sintética de C01 0.47 µm, C02 0.64 µm y C03 0.86 µm; NO existe canal verde ABI.'),
     ('B','C01, 0.47 µm: azul visible, sensible a dispersión atmosférica y nubosidad.')],
   'lectura':'Permite seguir nubes y plumas durante el día y comparar su movimiento con el terreno.',
   'limite':'La apariencia blanca o gris no diferencia por sí sola ceniza volcánica de nubes meteorológicas.'},
 'GeoColor aproximado (NO oficial)':{
   'canales':[
     ('R','Día: C02 0.64 µm. Noche: gradación térmica de C13, 10.3 µm.'),
     ('G','Día: verde sintético C01/C02/C03. Noche: gradación térmica C13.'),
     ('B','Día: C01 0.47 µm. Noche: C13 y contraste C07 3.9 µm − C13 10.3 µm.')],
   'lectura':'Facilita comparar la evolución de la nubosidad con un fondo visual familiar.',
   'limite':'Solo es una aproximación para clase; NO coincide con la receta operacional GeoColor de NOAA/CIRA.'},
 'Ash RGB':{
   'canales':[
     ('R','C15 − C13 (12.3 − 10.3 µm): diferencia split-window; sensible a absorción por ceniza frente a hielo.'),
     ('G','C14 − C11 (11.2 − 8.4 µm): contraste térmico y de propiedades de aerosoles/nubes.'),
     ('B','C13, 10.3 µm: temperatura de brillo; distingue superficies calientes y topes fríos.')],
   'lectura':'Algunas plumas de ceniza muestran magenta/rosa; seguir su continuidad y desplazamiento entre horas.',
   'limite':'Humedad, nubes y temperatura pueden producir colores parecidos. No es un detector inequívoco.'},
 'Fire Temperature RGB':{
   'canales':[
     ('R','C07, 3.9 µm: respuesta intensa a subpíxeles muy calientes, posibles puntos térmicos.'),
     ('G','C06, 2.2 µm: infrarrojo cercano de onda corta; ayuda a realzar calor intenso y diferencias superficiales.'),
     ('B','C05, 1.6 µm: reflectancia solar diurna, estructura de superficie y nubes.')],
   'lectura':'Una señal localizada puede indicar un foco térmico cerca del cráter si persiste y coincide espacialmente.',
   'limite':'El C07 recibe contribución solar durante el día; un píxel caliente no mide la temperatura exacta de lava.'},
 'Dust RGB':{
   'canales':[
     ('R','C15 − C13 (12.3 − 10.3 µm): respuesta a diferencia espectral de partículas.'),
     ('G','C14 − C11 (11.2 − 8.4 µm): separa ciertas estructuras minerales y nubosas.'),
     ('B','C13, 10.3 µm: temperatura de brillo, permite distinguir topes de nubes.')],
   'lectura':'Apoya la identificación de aerosoles minerales y su transporte; usa otros límites que Ash RGB.',
   'limite':'Polvo, ceniza y nubes pueden compartir respuesta espectral.'},
 'SO2 RGB':{
   'canales':[
     ('R','C09 − C10 (6.9 − 7.3 µm): contraste de bandas de vapor de agua.'),
     ('G','C13 − C11 (10.3 − 8.4 µm): diferencia con posible sensibilidad a SO₂ y composición de plumas.'),
     ('B','C07, 3.9 µm: señal térmica de la superficie y nube.')],
   'lectura':'Composición exploratoria para contrastar posibles plumas de gases con la nubosidad.',
   'limite':'No es recuperación cuantitativa de SO₂ ni un producto operacional certificado.'},
 'Night Microphysics RGB':{
   'canales':[
     ('R','C15 − C13 (12.3 − 10.3 µm): diferencias de emisividad y nubosidad.'),
     ('G','C13 − C07 (10.3 − 3.9 µm): contraste de nube, tamaño de gota y condiciones nocturnas.'),
     ('B','C13, 10.3 µm: temperatura de brillo de nube y superficie.')],
   'lectura':'Ayuda a interpretar nubosidad nocturna y distinguir estructuras que limitan la observación del cráter.',
   'limite':'No reemplaza Ash RGB ni confirma ceniza por sí solo.'},
 'Split Window C15−C13':{
   'canales':[('Diferencia','C15 12.3 µm menos C13 10.3 µm; se expresa en K de diferencia de temperatura de brillo.')],
   'lectura':'Algunos valores negativos sobre una pluma son compatibles con ceniza, dependiendo de la atmósfera.',
   'limite':'Es un indicador auxiliar: interpretar signo, magnitud y entorno, no usar umbral universal.'},
 'C07 3.9 µm (BT)':{
   'canales':[('Banda única','C07, 3.9 µm; temperatura de brillo en Kelvin.')],
   'lectura':'Permite buscar contrastes cálidos y seguir cambios cerca del cráter.',
   'limite':'Durante el día incluye componente solar reflejada. No equivale a radiancia Level 1b.'},
 'C13 10.3 µm (BT)':{
   'canales':[('Banda única','C13, 10.3 µm; temperatura de brillo en Kelvin.')],
   'lectura':'Topes fríos de nube/pluma frente a superficies cálidas; contexto de nubosidad.',
   'limite':'La temperatura de brillo no es necesariamente la temperatura física de la superficie.'},
}

# Se mantiene la función compute_products de la versión automática que ya funcionó.
# Esta representación conserva la explicación en el PNG, bajo la imagen.
def plot_layer(title,layer,notes,bounds,scene_time,path):
    ficha=FICHAS_RGB[title]
    fig=plt.figure(figsize=(8.2,9.3),facecolor='white')
    gs=fig.add_gridspec(2,1,height_ratios=[7.4,2.6],hspace=0.12)
    ax=fig.add_subplot(gs[0]);leg=fig.add_subplot(gs[1]);leg.axis('off')
    if layer.ndim==3:
        ax.imshow(np.nan_to_num(layer,nan=0,posinf=0,neginf=0),extent=(bounds[0],bounds[2],bounds[1],bounds[3]),origin='upper',interpolation='nearest')
    else:
        is_split='Split Window' in title
        im=ax.imshow(np.ma.masked_invalid(layer),extent=(bounds[0],bounds[2],bounds[1],bounds[3]),origin='upper',cmap='coolwarm' if is_split else 'inferno',
                     vmin=-10 if is_split else 190,vmax=10 if is_split else 340)
        fig.colorbar(im,ax=ax,shrink=.75,label='ΔBT (K)' if is_split else 'BT (K)')
    ax.set_title(f'{title}\n{SAT_ACTUAL} | {scene_time:%Y-%m-%d %H:%M} UTC',fontsize=11,fontweight='bold')
    ax.set_xlabel('Longitud (°)');ax.set_ylabel('Latitud (°)')
    leg.add_patch(plt.Rectangle((0,0),1,1,transform=leg.transAxes,facecolor='#f3f7fa',edgecolor='#d6e0e8'))
    leg.text(.03,.93,'Qué muestra cada canal',fontweight='bold',fontsize=10,va='top',transform=leg.transAxes)
    y=.78
    pal={'R':'#bb3535','G':'#16814a','B':'#2865b8','Diferencia':'#4c5561','Banda única':'#4c5561'}
    for canal,descr in ficha['canales']:
        wrapped=textwrap.fill(descr,width=100)
        leg.text(.04,y,canal+'  '+wrapped,color=pal.get(canal,'#233343'),fontsize=8.6,va='top',transform=leg.transAxes)
        y-=.16 if len(wrapped)<115 else .20
    leg.text(.04,max(.12,y-.02),'Observación: '+textwrap.fill(ficha['lectura'],width=100),fontsize=8.2,va='top',transform=leg.transAxes)
    leg.text(.04,.035,'Límite: '+textwrap.shorten(ficha['limite'],width=126,placeholder='…'),fontsize=7.7,color='#7f3929',va='bottom',transform=leg.transAxes)
    fig.savefig(path,dpi=128,bbox_inches='tight');plt.close(fig)

print('Fichas espectrales listas: se incluirán bajo cada producto.')

Fichas espectrales listas: se incluirán bajo cada producto.


## Paso 3. Seleccionar el volcán y la fecha

Se consulta automáticamente el repositorio público `noaa-goes19` de NOAA/AWS. El botón **GENERAR MAPAS** produce la escena más cercana a la hora elegida. Para seguir cambios a lo largo del tiempo, utiliza **ANIMAR 1–24 HORAS** en el paso siguiente. La descarga de NetCDF se realiza dentro de la sesión; no debes buscar ni subir archivos.

In [ ]:
from IPython.display import clear_output, Image

fecha=W.DatePicker(value=date(2026,11,14),description='Día UTC:')
hora=W.IntSlider(value=16,min=0,max=23,description='Hora UTC:',continuous_update=False)
minuto=W.IntSlider(value=20,min=0,max=50,step=10,description='Minuto:',continuous_update=False)
satelite_modo=W.Dropdown(
    options=[('Automático según fecha y longitud','auto'),('GOES-16 — Este histórico','16'),
             ('GOES-18 — Oeste','18'),('GOES-19 — Este actual','19')],
    value='auto',description='Satélite:',layout=W.Layout(width='430px'))

volcan=W.Dropdown(options=list(VOLCANES),value='Fuego',description='Volcán:')
lat=W.FloatText(value=VOLCANES['Fuego'][0],description='Latitud:')
lon=W.FloatText(value=VOLCANES['Fuego'][1],description='Longitud:')
radio=W.IntSlider(value=120,min=30,max=300,step=10,description='Radio km:',continuous_update=False)
capas=W.SelectMultiple(options=list(RGB_HELP),value=tuple(RGB_HELP),description='Productos:',rows=10,layout=W.Layout(width='430px',height='250px'))

def cambio_volcan(change):
    if change['name']=='value':lat.value,lon.value=VOLCANES[change['new']]
volcan.observe(cambio_volcan,names='value')

procesar=W.Button(description='DESCARGAR GOES Y GENERAR MAPAS',button_style='success',layout=W.Layout(width='350px'))
resultado=W.Output()
display(W.VBox([W.HTML('<b>Fecha y hora de referencia (UTC)</b>'),W.HBox([fecha,hora,minuto]),
                 W.HTML('<b>Área de estudio</b>'),W.HBox([volcan,lat,lon]),satelite_modo,radio,
                 W.HTML('<b>Capas a generar (todas seleccionadas por defecto)</b>'),capas,procesar,resultado]))

In [ ]:
def seleccionar_mcmipf():
    assert fecha.value is not None,'Selecciona un día UTC'
    target=datetime(fecha.value.year,fecha.value.month,fecha.value.day,hora.value,minuto.value,tzinfo=timezone.utc)
    horas=[h for h in [hora.value-1,hora.value,hora.value+1] if 0<=h<=23]
    tabla=s3_list(fecha.value,horas,product='ABI-L2-MCMIPF')
    if not tabla:
        # Una hora alternativa del mismo día ante huecos puntuales.
        tabla=s3_list(fecha.value,list(range(24)),product='ABI-L2-MCMIPF')
    if not tabla:raise FileNotFoundError('NOAA no devolvió escenas Full Disk del satélite seleccionado para este día')
    tabla.sort(key=lambda row:abs((row['fecha_utc']-target).total_seconds()))
    return tabla[0],target


def generar_mapas(_):
    global SAT_ACTUAL
    with resultado:
        clear_output(wait=True)
        procesar.disabled=True
        try:
            row,target=seleccionar_mcmipf()
            SAT_ACTUAL=_satellite_label(row)
            print('Fuente seleccionada:',SAT_ACTUAL)
            print('Escena más cercana:',row['fecha_utc'].strftime('%Y-%m-%d %H:%M:%S UTC'))
            print('Hora solicitada:',target.strftime('%Y-%m-%d %H:%M UTC'))
            path=descargar_automaticamente(row)
            tm=filename_time(path.name)
            req=[f'CMI_C{k:02d}' for k in (1,2,3,5,6,7,9,10,11,13,14,15)]
            with xr.open_dataset(path,engine='netcdf4',mask_and_scale=True) as ds:
                data,bounds=resample_local(ds,req,lat.value,lon.value,radio.value,n=220)
            products,es_dia=compute_products(data,tm,lat.value,lon.value)
            outdir=OUTPUT/f'MAPAS_{tm:%Y%m%d_%H%M}_{volcan.value}'
            outdir.mkdir(parents=True,exist_ok=True)
            pngs=[];creados=[];fallidos=[]
            for i,titulo in enumerate(capas.value):
                try:
                    arr=products[titulo]
                    if not np.isfinite(arr).any():
                        raise ValueError('No hay valores válidos para la región')
                    basename=re.sub(r'[^A-Za-z0-9]+','_',titulo).strip('_')
                    png=outdir/f'{i:02d}_{basename}.png'
                    tif=png.with_suffix('.tif')
                    plot_layer(titulo,arr,RGB_HELP[titulo],bounds,tm,png)
                    save_geotiff(tif,arr,bounds)
                    creados.extend([png,tif]);pngs.append(png)
                    print('OK:',titulo)
                except Exception as err:
                    fallidos.append({'producto':titulo,'error':str(err)})
                    print('NO DISPONIBLE:',titulo,str(err))
            # Estadística exploratoria: cráter vs anillo, C07 es BT en K, no radiancia L1b.
            if 'CMI_C07' in data:
                temp=data['CMI_C07']
                x=np.linspace(bounds[0],bounds[2],temp.shape[1])
                y=np.linspace(bounds[3],bounds[1],temp.shape[0])
                X,Y=np.meshgrid(x,y)
                dx=(X-lon.value)*111.32*np.cos(np.deg2rad(lat.value))
                dy=(Y-lat.value)*111.32
                dist=np.sqrt(dx*dx+dy*dy)
                core=temp[(dist<=5)&np.isfinite(temp)]
                bg=temp[(dist>=10)&(dist<=20)&np.isfinite(temp)]
                if core.size and bg.size:
                    delta=float(np.max(core)-np.median(bg))
                    print(f'C07: BT máxima cráter={float(np.max(core)):.2f} K; mediana entorno={float(np.median(bg)):.2f} K; contraste={delta:+.2f} K')
                    print('Contraste térmico exploratorio: no demuestra lava ni constituye alerta oficial.')
                else:
                    print('C07: no hay suficientes píxeles para contraste cráter-entorno.')
            if pngs:
                gal=outdir/'GALERIA_DOS_COLUMNAS.jpg'
                make_gallery(pngs).save(gal,quality=92)
                creados.append(gal)
                display(Image(filename=str(gal),width=1050))
            meta=outdir/'metadatos.json'
            meta.write_text(json.dumps({'fuente':SAT_ACTUAL+' NOAA AWS público','archivo_NOAA':path.name,
                'fecha_utc':str(tm),'hora_solicitada_utc':str(target),'volcan':volcan.value,
                'lat':lat.value,'lon':lon.value,'radio_km':radio.value,'iluminacion_diurna':bool(es_dia),
                'geocolor':'Aproximación didáctica; no producto operacional oficial',
                'ceniza':'Ash RGB ayuda a interpretar, no confirma ceniza de forma automática',
                'fallidos':fallidos},ensure_ascii=False,indent=2))
            creados.append(meta)
            zipfile_path=outdir/'GOES_MAPAS_Y_GEOTIFF.zip'
            with zipfile.ZipFile(zipfile_path,'w',zipfile.ZIP_DEFLATED) as z:
                for f in creados:z.write(f,arcname=f.name)
            print(f'Productos listos: {len(pngs)} / {len(capas.value)}. Paquete final: {zipfile_path.name}')
            mostrar_descarga(zipfile_path)
        except Exception as err:
            print('No se pudo completar la consulta pública de NOAA:',repr(err))
            print('Verifica conexión de Colab, fecha UTC y disponibilidad NOAA. No es necesario subir archivos.')
        finally:
            procesar.disabled=False

procesar.on_click(generar_mapas)
print('Listo. Selecciona fecha/hora y pulsa el botón verde; no necesitas subir nada.')

Listo. Selecciona fecha/hora y pulsa el botón verde; no necesitas subir nada.


## Paso 4. Interpretación de los mapas

- **Ash RGB:** aporta contrastes compatibles con partículas de ceniza; no supone una detección automática confirmada.
- **Fire Temperature:** un foco pequeño y persistente en el cráter es un candidato térmico, pero requiere verificación y puede estar afectado por reflejos solares, nubes, saturación o geolocalización.
- **True Color / GeoColor:** se interpretan principalmente en condiciones diurnas; el GeoColor incluido aquí es una aproximación educativa.
- **SO₂ RGB:** composición exploratoria, no mide concentración de gas.

Los límites y longitudes de onda aparecen **debajo de cada imagen exportada**.

## Paso 5. Animación automática GOES-19 (1 a 4 horas)

Elige el intervalo UTC, su duración (**mínimo una hora, máximo cuatro**) y el paso entre fotogramas (10, 20 o 30 minutos). Se eligen las escenas más cercanas a la cuadrícula horaria, sin repetir un mismo archivo.

Para cuidar la memoria de Colab, se recomienda comenzar con **1 hora, cada 20 minutos y 3 RGB**. A cuatro horas con 10 minutos pueden procesarse hasta **25 fotogramas** y el volumen de descarga será mayor. Los archivos NetCDF temporales se borran después de cada escena, a menos que se habilite explícitamente conservarlos. Las salidas incluyen GIF individuales, una animación conjunta en dos columnas, CSV de fotogramas y ZIP.

**Recuerda:** animar la pluma permite analizar desplazamiento temporal; no determina por sí solo altura, composición ni concentración.

In [ ]:
from PIL import Image as PILImage, ImageDraw, ImageFont
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from IPython.display import Image as IPyImage, HTML as DisplayHTML

anim_duracion=W.IntSlider(value=2,min=1,max=24,step=1,description='Horas:',continuous_update=False)
anim_paso=W.Dropdown(options=[('10 min (máximo detalle)',10),('20 min (recomendado)',20),('30 min (ligero)',30),('60 min (muy ligero)',60)],value=20,description='Paso:')
anim_capas=W.SelectMultiple(options=list(RGB_HELP),value=('True Color aproximado','Ash RGB','Fire Temperature RGB'),description='Capas:',rows=10,
                            layout=W.Layout(width='470px',height='230px'))
anim_guardar=W.Checkbox(value=False,description='Conservar archivos NetCDF temporales (más espacio)')
anim_boton=W.Button(description='ANIMAR GOES (1–24 HORAS)',button_style='success',layout=W.Layout(width='330px'))
anim_salida=W.Output()

display(W.HTML('<b>Animación: fecha, hora y volcán se toman de la sección anterior.</b><br><span style="color:#444">Si el punto está en Guatemala, se muestran límites departamentales. En otros países, se muestra el contorno nacional.</span>'))
display(W.VBox([W.HBox([anim_duracion,anim_paso]),W.HBox([anim_capas,anim_guardar]),anim_boton,anim_salida]))

HTML(value='<b>Animación: fecha, hora y volcán se toman de la sección anterior.</b><br><span style="color:#444…

In [ ]:
def _hours_for_times(dt0,dt1):
    d0=dt0.replace(minute=0,second=0,microsecond=0)-timedelta(hours=1)
    d1=dt1.replace(minute=0,second=0,microsecond=0)+timedelta(hours=1)
    ans={}
    t=d0
    while t<=d1:
        ans.setdefault(t.date(),set()).add(t.hour)
        t+=timedelta(hours=1)
    return ans


def _select_animation_rows(start,duration_h,step_minutes):
    end=start+timedelta(hours=int(duration_h))
    slots=[start+timedelta(minutes=i*step_minutes) for i in range(int(duration_h*60//step_minutes)+1)]
    candidate=[]
    for day,hours in _hours_for_times(start,end).items():
        candidate.extend(s3_list(day,sorted(hours),product='ABI-L2-MCMIPF'))
    candidates=sorted(candidate,key=lambda z:z['fecha_utc'])
    if not candidates:
        raise FileNotFoundError('No se hallaron escenas del satélite seleccionado para este período UTC')
    result=[];used=set()
    for target in slots:
        ordered=sorted(candidates,key=lambda r:abs((r['fecha_utc']-target).total_seconds()))
        for row in ordered:
            if abs((row['fecha_utc']-target).total_seconds())<=8*60 and row['key'] not in used:
                used.add(row['key']);result.append(row);break
    if len(result)<2: raise ValueError('Menos de dos escenas próximas a los horarios solicitados. Cambia hora o fecha.')
    return result


def _font(sz=16,bold=False):
    names=['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf']
    for p in names:
        if Path(p).exists(): return ImageFont.truetype(p,sz)
    return ImageFont.load_default()


def _frame_for_layer(name,arr,bounds,timestamp,size=(500,500)):
    # El panel no usa estiramientos temporales nuevos: compute_products mantiene escalas fijas.
    fig,ax=plt.subplots(figsize=(5.4,5.2),dpi=100)
    if arr.ndim==3:
        ax.imshow(np.nan_to_num(arr,nan=0,posinf=0,neginf=0),extent=(bounds[0],bounds[2],bounds[1],bounds[3]),origin='upper')
    else:
        split='Split Window' in name
        im=ax.imshow(np.ma.masked_invalid(arr),extent=(bounds[0],bounds[2],bounds[1],bounds[3]),origin='upper',cmap='coolwarm' if split else 'inferno',vmin=-10 if split else 190,vmax=10 if split else 340)
        fig.colorbar(im,ax=ax,label='ΔBT (K)' if split else 'BT (K)',shrink=.76)
    draw_reference_boundaries(ax, lat.value if 'lat' in globals() else 14.47, lon.value if 'lon' in globals() else -90.88, bounds)
    ax.set_xlim(bounds[0],bounds[2]); ax.set_ylim(bounds[1],bounds[3])
    ax.set_title(name+'\n'+timestamp.strftime('%Y-%m-%d %H:%M UTC'),fontsize=10)
    ax.set_xlabel('Longitud (°)',fontsize=8);ax.set_ylabel('Latitud (°)',fontsize=8)
    ax.tick_params(labelsize=7)
    fig.tight_layout();buffer=BytesIO();fig.savefig(buffer,format='png',dpi=95);plt.close(fig)
    buffer.seek(0)
    with PILImage.open(buffer) as im:
        im=im.convert('RGB').resize(size, PILImage.Resampling.LANCZOS)
    return im


def _combine_two_cols(items, names,tm):
    if not names: return None
    selected=[items.get(n) for n in names]
    tile_w,tile_h=500,602
    cols=2; rows=math.ceil(len(names)/cols)
    panel=PILImage.new('RGB',(cols*tile_w,rows*tile_h+46),'#ffffff')
    d=ImageDraw.Draw(panel);d.rectangle((0,0,panel.width,45),fill='#12374c')
    d.text((14,13),f'INSIVUMEH · {SAT_ACTUAL} · {tm:%Y-%m-%d %H:%M UTC}',fill='white',font=_font(16,True))
    for i,name in enumerate(names):
        x=(i%cols)*tile_w;y=46+(i//cols)*tile_h
        panel.paste(items[name],(x,y))
        d.text((x+12,y+503),name[:45],fill='#21364a',font=_font(12,True))
        ficha=FICHAS_RGB[name]
        line_y=y+522
        for canal,descripcion in ficha['canales'][:3]:
            d.text((x+12,line_y),textwrap.shorten(canal+': '+descripcion,width=78,placeholder='…'),
                   fill='#27546b',font=_font(10))
            line_y+=17
        d.text((x+12,line_y+1),textwrap.shorten('Interpretación: '+ficha['lectura'],width=76,placeholder='…'),
               fill='#5b3d27',font=_font(10))
    return panel


def _gif_save(frames,path,ms=550):
    if not frames: return
    # Cuantización individual compatible con GIF; la paleta puede variar levemente.
    q=[f.convert('P',palette=PILImage.Palette.ADAPTIVE,colors=224) for f in frames]
    q[0].save(str(path),save_all=True,append_images=q[1:],duration=ms,loop=0,optimize=True,disposal=2)


def ejecutar_animacion(_):
    global SAT_ACTUAL
    with anim_salida:
        clear_output(wait=True)
        anim_boton.disabled=True
        try:
            assert fecha.value is not None,'Selecciona día UTC'
            assert len(anim_capas.value)>0,'Selecciona al menos una capa'
            if len(anim_capas.value)>6:
                raise ValueError('Para evitar archivos muy grandes, selecciona hasta 6 capas animadas a la vez')
            start=datetime(fecha.value.year,fecha.value.month,fecha.value.day,hora.value,minuto.value,tzinfo=timezone.utc)
            tabla=_select_animation_rows(start,anim_duracion.value,anim_paso.value)
            print('Escenas seleccionadas:',len(tabla), '| ventana:',anim_duracion.value,'h | paso:',anim_paso.value,'min')
            total_mb=sum(row['MB'] for row in tabla)
            print(f'Descarga nominal aproximada: {total_mb:.1f} MB; las imágenes completas se procesan en recortes.')
            print('Los datos se obtienen directamente de NOAA/AWS. No hay que cargar archivos.')
            folder=OUTPUT/f'ANIMACION_{start:%Y%m%d_%H%M}_{int(anim_duracion.value)}h_{volcan.value}'
            folder.mkdir(parents=True,exist_ok=True)
            names=list(anim_capas.value)
            frames={name:[] for name in names}
            multiview=[];log=[];fails=[]
            req=[f'CMI_C{k:02d}' for k in (1,2,3,5,6,7,9,10,11,13,14,15)]
            for j,row in enumerate(tabla,1):
                local=None
                try:
                    print(f'[{j}/{len(tabla)}] {row["fecha_utc"]:%Y-%m-%d %H:%M} UTC')
                    SAT_ACTUAL=_satellite_label(row)
                    local=descargar_automaticamente(row)
                    with xr.open_dataset(local,engine='netcdf4',mask_and_scale=True) as ds:
                        data,bounds=resample_local(ds,req,lat.value,lon.value,radio.value,n=200)
                    products,daylight=compute_products(data,row['fecha_utc'],lat.value,lon.value)
                    frame_items={}
                    for name in names:
                        frame=_frame_for_layer(name,products[name],bounds,row['fecha_utc'])
                        frames[name].append(frame);frame_items[name]=frame
                    multiview.append(_combine_two_cols(frame_items,names,row['fecha_utc']))
                    log.append({'fecha_UTC':str(row['fecha_utc']),'satelite':SAT_ACTUAL,'archivo':Path(row['key']).name,'tamaño_MB':row['MB'],
                                'iluminacion_diurna':bool(daylight)})
                except Exception as ex:
                    fails.append({'fecha_UTC':str(row['fecha_utc']),'error':str(ex)})
                    print('  Escena omitida:',repr(ex))
                finally:
                    if local is not None and not anim_guardar.value:
                        try: local.unlink(missing_ok=True)
                        except Exception: pass
            if len(log)<2:
                raise RuntimeError('No se pudieron procesar al menos dos fotogramas; consulta el registro de errores')
            created=[]
            for name,imgs in frames.items():
                fname=re.sub(r'[^A-Za-z0-9]+','_',name).strip('_')
                gif=folder/f'GOES_{fname}.gif';_gif_save(imgs,gif)
                created.append(gif)
            gif=folder/'GOES_COMPARACION_2_COLUMNAS.gif'
            _gif_save(multiview,gif)
            created.append(gif)
            pd.DataFrame(log).to_csv(folder/'fotogramas.csv',index=False)
            created.append(folder/'fotogramas.csv')
            info=folder/'leyendas_espectrales.json'
            info.write_text(json.dumps({'capas':{k:FICHAS_RGB[k] for k in names},'errores':fails,'fuente':'noaa-goes19 AWS pública',
                              'mensaje':'Productos de interpretación, no detección de ceniza confirmada.'},ensure_ascii=False,indent=2))
            created.append(info)
            paquete=folder/'GOES_ANIMACIONES_Y_EXPLICACIONES.zip'
            with zipfile.ZipFile(paquete,'w',zipfile.ZIP_DEFLATED) as z:
                for p in created:z.write(p,arcname=p.name)
            print(f'Fotogramas válidos: {len(log)} | omitidos: {len(fails)} | animaciones: {len(names)} + comparación 2 columnas')
            display(IPyImage(filename=str(gif),width=900))
            explicaciones=['<h3>Interpretación de las capas animadas</h3>']
            for name in names:
                ficha=FICHAS_RGB[name]
                explicaciones.append('<h4>'+name+'</h4>')
                explicaciones.extend('<p><b>'+canal+':</b> '+descripcion+'</p>' for canal,descripcion in ficha['canales'])
                explicaciones.append('<p><b>Qué observar:</b> '+ficha['lectura']+'</p>')
                explicaciones.append('<p><i>Limitación: '+ficha['limite']+'</i></p>')
            display(DisplayHTML(''.join(explicaciones)))
            print('Las fichas también se incluyen en leyendas_espectrales.json y en los mapas estáticos.')
            mostrar_descarga(paquete)
        except Exception as ex:
            print('No se completó la animación:',repr(ex))
            print('Comprueba fecha/hora UTC y disponibilidad de NOAA; no debes buscar archivos manualmente.')
        finally:
            anim_boton.disabled=False

anim_boton.on_click(ejecutar_animacion)
print('Animación lista: seleccionar 1–24 horas y pulsar el botón verde.')

Animación lista: seleccionar 1–24 horas y pulsar el botón verde.


## Comparación GOES-19 ↔ MODIS (referencia científica MODVOLC)

**MODVOLC**, desarrollado por la Universidad de Hawái, identifica anomalías térmicas volcánicas usando observaciones MODIS de Terra y Aqua. Sus *text alert files* incluyen hora, ubicación y radiancias espectrales de píxeles detectados. Es distinto de la temperatura de brillo C07 de GOES-19. [NASA: MODVOLC](https://modis-land.gsfc.nasa.gov/modvolc.html).

**Disponibilidad de datos:** el sitio de MODVOLC ofrece consultas y archivos de texto, pero no se pudo verificar una interfaz de descarga programática por fecha. Para no inventar datos MODVOLC, este módulo toma **detecciones MODIS Collection 6.1 de NASA FIRMS** (algoritmo independiente) como referencia contrastable, y muestra **MODIS/FIRMS** en todas las salidas. En general, puede consultar automáticamente el CSV público **de los últimos 7 días**. El acceso al archivo histórico de FIRMS requiere autenticación/solicitud y no se sustituye con fechas falsas. El cuaderno no solicita archivos ni claves.

**Interpretación:** MODIS/FIRMS proporciona fecha/hora, coordenadas, temperatura de brillo 4 μm (K) y, si está disponible, FRP (MW). GOES-19 proporciona BT C07 en K o radiancia Level 1b (según el cuaderno). **FRP (MW), BT (K) y radiancia (unidades espectrales) no son magnitudes equivalentes**; no se deben sumar ni usar un mismo eje vertical. MODVOLC y FIRMS no tienen exactamente los mismos umbrales ni las mismas detecciones.


In [ ]:
# MODIS/FIRMS · referencia térmica independiente de GOES-19 (NO son alertas MODVOLC)
import requests
from io import StringIO
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIRMS_MODIS_URL = 'https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/csv/MODIS_C6_1_Global_7d.csv'
FIRMS_CACHE = DATA / 'MODIS_C6_1_Global_7d.csv'
FIRMS_CACHE_META = DATA / 'MODIS_C6_1_Global_7d_descarga_UTC.txt'

def _modis_temporal_disponible(start, end):
    now = datetime.now(timezone.utc)
    return start <= now and end >= now - timedelta(days=7)

def _haversine_modis(lat1, lon1, lat2, lon2):
    r = 6371.0
    dlat = np.deg2rad(np.asarray(lat2, dtype=float) - float(lat1))
    dlon = np.deg2rad(np.asarray(lon2, dtype=float) - float(lon1))
    a = np.sin(dlat/2)**2 + np.cos(np.deg2rad(lat1)) * np.cos(np.deg2rad(lat2)) * np.sin(dlon/2)**2
    return 2*r*np.arcsin(np.minimum(1, np.sqrt(a)))

def obtener_modis_firms_publico(start, end, centro_lat, centro_lon, radio_km=25, max_cache_h=6):
    """CSV público MODIS FIRMS (últimos 7 días); no consulta MODVOLC.
    Retorna (df, status). Usa solo CSV real, nunca genera detecciones sintéticas."""
    empty = pd.DataFrame(columns=['fecha_UTC', 'latitude', 'longitude', 'frp', 'brightness', 'satellite', 'distancia_km'])
    now = datetime.now(timezone.utc)
    if not _modis_temporal_disponible(start, end):
        return empty, 'Fuera de la ventana de los últimos 7 días. No hay consulta pública automática para la fecha seleccionada.'
    if start < now-timedelta(days=7) or end > now:
        warning = 'Solo se mostrarán observaciones de la porción disponible de los últimos 7 días.'
    else:
        warning = ''
    # El CSV global se lee por fragmentos para no cargar todo el planeta en la RAM de Colab.
    cache_ok = (FIRMS_CACHE.exists() and FIRMS_CACHE.stat().st_size > 100 and
                (now-datetime.fromtimestamp(FIRMS_CACHE.stat().st_mtime, timezone.utc)).total_seconds() < max_cache_h*3600)
    if not cache_ok:
        partial = FIRMS_CACHE.with_suffix('.part')
        try:
            with requests.get(FIRMS_MODIS_URL, stream=True, timeout=(20,120),
                              headers={'User-Agent':'GOES19-MODIS-educativo/1.0'}) as res:
                res.raise_for_status()
                if 'html' in res.headers.get('Content-Type','').lower():
                    raise IOError('El servidor devolvió HTML en lugar de CSV')
                with open(partial, 'wb') as f:
                    for chunk in res.iter_content(chunk_size=1024*512):
                        if chunk: f.write(chunk)
            if partial.stat().st_size < 100:
                raise IOError('CSV demasiado pequeño; respuesta incompleta')
            partial.replace(FIRMS_CACHE)
            FIRMS_CACHE_META.write_text('Descarga UTC: '+str(now)+'\nURL: '+FIRMS_MODIS_URL, encoding='utf-8')
        except Exception as ex:
            partial.unlink(missing_ok=True)
            if not FIRMS_CACHE.exists():
                return empty, f'FIRMS no disponible sin autenticación: {type(ex).__name__}: {str(ex)[:180]}'
            warning += ' Se reutilizó una copia en caché por fallo de descarga; comprueba su fecha.'
    pieces=[]
    try:
        for block in pd.read_csv(FIRMS_CACHE, chunksize=120_000, low_memory=False):
            block.columns=[str(x).strip().lower() for x in block.columns]
            if not {'latitude','longitude','acq_date','acq_time'}.issubset(block.columns):
                return empty, 'El CSV recibido no contiene las columnas MODIS esperadas.'
            lat_v=pd.to_numeric(block['latitude'],errors='coerce')
            lon_v=pd.to_numeric(block['longitude'],errors='coerce')
            # Filtro rectangular previo para ahorrar cálculos
            mask=lat_v.between(centro_lat-radio_km/110, centro_lat+radio_km/110) & \
                 lon_v.between(centro_lon-radio_km/(111*max(.1,np.cos(np.deg2rad(centro_lat)))),
                               centro_lon+radio_km/(111*max(.1,np.cos(np.deg2rad(centro_lat)))))
            if not mask.any(): continue
            sub=block.loc[mask].copy()
            sub['latitude']=lat_v.loc[mask].values;sub['longitude']=lon_v.loc[mask].values
            sub['distancia_km']=_haversine_modis(centro_lat,centro_lon,sub['latitude'],sub['longitude'])
            sub=sub.loc[sub['distancia_km']<=radio_km].copy()
            if sub.empty: continue
            acqtime=sub['acq_time'].astype(str).str.split('.').str[0].str.zfill(4)
            sub['fecha_UTC']=pd.to_datetime(sub['acq_date'].astype(str)+' '+acqtime,
                                              format='%Y-%m-%d %H%M',utc=True,errors='coerce')
            sub=sub.loc[(sub['fecha_UTC']>=start)&(sub['fecha_UTC']<=end)].copy()
            if sub.empty: continue
            for name in ('frp','brightness','bright_t31','confidence'):
                if name in sub:
                    sub[name]=pd.to_numeric(sub[name],errors='coerce')
            pieces.append(sub)
    except Exception as ex:
        return empty, f'No se pudo interpretar CSV MODIS/FIRMS: {type(ex).__name__}: {ex}'
    if not pieces: return empty, 'No hay detecciones MODIS/FIRMS en el radio y fecha consultados. '+warning
    result=pd.concat(pieces,ignore_index=True).sort_values('fecha_UTC')
    return result, f'{len(result)} detecciones MODIS/FIRMS · radio {radio_km:.0f} km. '+warning

def fig_mapa_firms(df, centro_lat, centro_lon, radio_km=25, titulo='MODIS/FIRMS · detecciones térmicas'):
    fig,ax=plt.subplots(figsize=(8,7))
    dlat=radio_km/111.3;dlon=radio_km/(111.3*max(.15,np.cos(np.deg2rad(centro_lat))))
    ax.set_xlim(centro_lon-dlon,centro_lon+dlon)
    ax.set_ylim(centro_lat-dlat,centro_lat+dlat)
    if not df.empty:
        if 'frp' in df and pd.to_numeric(df['frp'],errors='coerce').notna().any():
            frp=pd.to_numeric(df['frp'],errors='coerce').fillna(0)
            sizes=35+np.clip(frp,0,150)*1.8
        else: sizes=np.full(len(df),45)
        ax.scatter(df['longitude'],df['latitude'],s=sizes,c='orangered',alpha=.8,
                   edgecolors='white',linewidths=.7,label='MODIS/FIRMS',zorder=6)
    ax.scatter([centro_lon],[centro_lat],marker='^',s=150,c='darkred',edgecolors='white',label='Volcán / AOI',zorder=8)
    ax.set_aspect(1/max(.15,np.cos(np.deg2rad(centro_lat))))
    ax.set_title(titulo)
    ax.set_xlabel('Longitud (°)');ax.set_ylabel('Latitud (°)')
    ax.grid(alpha=.15);ax.legend(loc='upper right')
    fig.tight_layout()
    return fig


### Comparar el volcán y la fecha seleccionada

Pulsa el botón **COMPARAR MODIS/FIRMS + GOES-19**. El programa intenta descargar las detecciones MODIS/FIRMS de NASA, calcula distancias al volcán, dibuja su ubicación y representa el canal 7 GOES-19 más cercano a la hora elegida. Si la fecha ya no está en el intervalo público de siete días, dibuja solo el contexto de GOES e informa la limitación; no inventa MODVOLC.


In [ ]:
from IPython.display import clear_output, display, Image
cmp_radio=W.IntSlider(value=25,min=5,max=75,step=5,description='Radio km:',continuous_update=False)
cmp_btn=W.Button(description='COMPARAR MODIS/FIRMS + GOES',button_style='info',layout=W.Layout(width='360px'))
cmp_out=W.Output()
display(W.VBox([cmp_radio,cmp_btn,cmp_out]))

def _ejecutar_comparacion(_):
    with cmp_out:
        clear_output(wait=True)
        cmp_btn.disabled=True
        try:
            assert fecha.value is not None, 'Selecciona una fecha UTC'
            target=datetime(fecha.value.year,fecha.value.month,fecha.value.day,hora.value,minuto.value,tzinfo=timezone.utc)
            day0=target.replace(hour=0,minute=0,second=0)
            day1=day0+timedelta(days=1)-timedelta(seconds=1)
            df,status=obtener_modis_firms_publico(day0,day1,lat.value,lon.value,cmp_radio.value)
            print('NASA MODIS/FIRMS:',status)
            folder=OUTPUT/f'COMPARACION_GOES19_MODIS_{target:%Y%m%d_%H%M}_{volcan.value}'
            folder.mkdir(parents=True,exist_ok=True)
            df.to_csv(folder/'detecciones_MODIS_FIRMS.csv',index=False)
            fig=fig_mapa_firms(df,lat.value,lon.value,cmp_radio.value,
                               titulo=f'MODIS/FIRMS · {target:%Y-%m-%d} · {volcan.value}')
            fig.savefig(folder/'MODIS_FIRMS_mapa.png',dpi=155)
            display(fig);plt.close(fig)
            if not df.empty:
                print('Muestras:',len(df),' | horas MODIS:',df['fecha_UTC'].dt.strftime('%H:%M').unique().tolist())
                display(df[['fecha_UTC','latitude','longitude']+[k for k in ['frp','brightness','satellite'] if k in df]].head(30))
            # Cargar C07 real de GOES (sin confundir la señal con radiancia L1b)
            row,ref=seleccionar_mcmipf()
            archive=descargar_automaticamente(row)
            with xr.open_dataset(archive,engine='netcdf4',mask_and_scale=True) as ds:
                bands,bounds=resample_local(ds,['CMI_C07'],lat.value,lon.value,max(cmp_radio.value,radio.value),n=220)
            img=bands['CMI_C07']
            fig,ax=plt.subplots(figsize=(8,7))
            im=ax.imshow(np.ma.masked_invalid(img),extent=(bounds[0],bounds[2],bounds[1],bounds[3]),
                         origin='upper',cmap='inferno',vmin=190,vmax=340)
            fig.colorbar(im,ax=ax,label='GOES-19 C07 · temperatura de brillo (K)')
            if not df.empty:
                ax.scatter(df['longitude'],df['latitude'],s=55,c='cyan',marker='o',edgecolors='black',
                           linewidths=.7,label='Detecciones MODIS/FIRMS',zorder=9)
            ax.scatter([lon.value],[lat.value],marker='^',s=140,c='red',edgecolors='white',zorder=12,label='Volcán / AOI')
            draw_reference_boundaries(ax,lat.value,lon.value,bounds)
            ax.set_xlim(bounds[0],bounds[2]);ax.set_ylim(bounds[1],bounds[3])
            ax.set_title(f'GOES-19 C07 y MODIS/FIRMS | {row["fecha_utc"]:%Y-%m-%d %H:%M} UTC')
            ax.set_xlabel('Longitud (°)');ax.set_ylabel('Latitud (°)');ax.legend(loc='upper right')
            fig.tight_layout();fig.savefig(folder/'GOES_C07_MODIS_superpuestos.png',dpi=160)
            display(fig);plt.close(fig)
            print('Nota: no hay coincidencia temporal exacta necesariamente. No mezclar GOES C07 (K) con FRP (MW).')
            zip_path=folder/'GOES19_MODIS_COMPARACION.zip'
            with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
                for p in folder.iterdir():
                    if p.is_file() and p!=zip_path:z.write(p,arcname=p.name)
            mostrar_descarga(zip_path)
        except Exception as e:
            print('No se completó la comparación:',repr(e))
        finally:
            cmp_btn.disabled=False

cmp_btn.on_click(_ejecutar_comparacion)
